Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [5]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [6]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [7]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [8]:
datosNormalizados.shape

(52416, 5)

In [9]:
datosNormalizados.head(13)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [10]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [11]:
futuros = 1
pasados  = 12

In [12]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [13]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 5)
Dimensiones de Y: (52404, 1)


In [14]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012]]


Se dividen nuevamente los conjuntos de datos

In [15]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 5)
Las dimensiones de testX son:  (10533, 12, 5)
Las dimensiones de valX son:  (5189, 12, 5)


In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [17]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [18]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [19]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [20]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

287/287 - 16s - 57ms/step - ia: 0.3144 - loss: 2.2542 - mae: 1.2336 - rmse: 1.4856 - smape: 1.4094 - val_ia: 0.2939 - val_loss: 1.2548 - val_mae: 0.9354 - val_rmse: 1.0983 - val_smape: 1.4434

Epoch 2/128                                           

287/287 - 9s - 31ms/step - ia: 0.2528 - loss: 1.2654 - mae: 0.9270 - rmse: 1.1229 - smape: 1.5021 - val_ia: 0.2124 - val_loss: 0.8966 - val_mae: 0.7884 - val_rmse: 0.9384 - val_smape: 1.7010

Epoch 3/128                                           

287/287 - 5s - 18ms/step - ia: 0.2284 - loss: 1.1448 - mae: 0.8848 - rmse: 1.0687 - smape: 1.5332 - val_ia: 0.2177 - val_loss: 0.8551 - val_mae: 0.7695 - val_rmse: 0.9182 - val_smape: 1.9006

Epoch 4/128                                           

287/287 - 5s - 17ms/step - ia: 0.2210 - loss: 1.0774 - mae: 0.8591 - rmse: 1.0363 - smape: 1.5438 - val_ia: 0.2319 - val_loss: 0.8236 - val_mae: 0.7545 - val_rmse: 0.9012 - val_smape: 1.8427

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

2293/2293 - 63s - 27ms/step - ia: 0.9544 - loss: 0.0147 - mae: 0.0699 - rmse: 0.0929 - smape: 0.1998 - val_ia: 0.7910 - val_loss: 0.0044 - val_mae: 0.0516 - val_rmse: 0.0624 - val_smape: 0.1447

Epoch 2/128                                                                       

2293/2293 - 50s - 22ms/step - ia: 0.9731 - loss: 0.0038 - mae: 0.0419 - rmse: 0.0559 - smape: 0.1408 - val_ia: 0.8090 - val_loss: 0.0033 - val_mae: 0.0434 - val_rmse: 0.0540 - val_smape: 0.1367

Epoch 3/128                                                                       

2293/2293 - 83s - 36ms/step - ia: 0.9740 - loss: 0.0036 - mae: 0.0406 - rmse: 0.0544 - smape: 0.1381 - val_ia: 0.8186 - val_loss: 0.0033 - val_mae: 0.0422 - val_rmse: 0.0522 - val_smape: 0.1319

Epoch 4/128                                                                       

2293/2293 - 51s - 22ms/step - ia: 0.9744 - loss: 0.0035 - mae: 0.0399 - rmse: 0

...

Epoch 1/128                                                                            

2293/2293 - 30s - 13ms/step - ia: 0.6312 - loss: 0.4675 - mae: 0.5163 - rmse: 0.6319 - smape: 0.9205 - val_ia: 0.4056 - val_loss: 0.1455 - val_mae: 0.3029 - val_rmse: 0.3284 - val_smape: 0.6451

Epoch 2/128                                                                            

2293/2293 - 26s - 11ms/step - ia: 0.8152 - loss: 0.1317 - mae: 0.2848 - rmse: 0.3555 - smape: 0.5929 - val_ia: 0.4537 - val_loss: 0.1035 - val_mae: 0.2569 - val_rmse: 0.2760 - val_smape: 0.5891

Epoch 3/128                                                                            

2293/2293 - 27s - 12ms/step - ia: 0.8551 - loss: 0.0830 - mae: 0.2238 - rmse: 0.2818 - smape: 0.4984 - val_ia: 0.5268 - val_loss: 0.0600 - val_mae: 0.1948 - val_rmse: 0.2112 - val_smape: 0.4989

Epoch 4/128                                                                            

2293/2293 - 24s - 10ms/step - ia: 0.8813 - loss: 0.0572 - m

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                             

4586/4586 - 56s - 12ms/step - ia: 0.3097 - loss: 0.8850 - mae: 0.7620 - rmse: 0.9176 - smape: 1.5007 - val_ia: 0.1448 - val_loss: 0.7894 - val_mae: 0.7431 - val_rmse: 0.7597 - val_smape: 1.7577

Epoch 2/128                                                                             

4586/4586 - 36s - 8ms/step - ia: 0.3342 - loss: 0.8310 - mae: 0.7358 - rmse: 0.8898 - smape: 1.4587 - val_ia: 0.1472 - val_loss: 0.7455 - val_mae: 0.7232 - val_rmse: 0.7394 - val_smape: 1.7026

Epoch 3/128                                                                             

4586/4586 - 39s - 8ms/step - ia: 0.3581 - loss: 0.7839 - mae: 0.7127 - rmse: 0.8621 - smape: 1.4223 - val_ia: 0.1498 - val_loss: 0.7031 - val_mae: 0.7033 - val_rmse: 0.7191 - val_smape: 1.6365

Epoch 4/128                                                                             

4586/4586 - 38s - 8ms/step - ia: 0.3796 - loss: 0.7413 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                             

287/287 - 8s - 27ms/step - ia: 0.6161 - loss: 0.4954 - mae: 0.5357 - rmse: 0.6602 - smape: 0.9657 - val_ia: 0.7755 - val_loss: 0.1455 - val_mae: 0.3013 - val_rmse: 0.3781 - val_smape: 0.6704

Epoch 2/128                                                                             

287/287 - 4s - 13ms/step - ia: 0.7923 - loss: 0.1798 - mae: 0.3357 - rmse: 0.4225 - smape: 0.6772 - val_ia: 0.7993 - val_loss: 0.1181 - val_mae: 0.2769 - val_rmse: 0.3384 - val_smape: 0.6287

Epoch 3/128                                                                             

287/287 - 4s - 13ms/step - ia: 0.8260 - loss: 0.1292 - mae: 0.2833 - rmse: 0.3582 - smape: 0.6091 - val_ia: 0.8198 - val_loss: 0.0953 - val_mae: 0.2508 - val_rmse: 0.2982 - val_smape: 0.5946

Epoch 4/128                                                                             

287/287 - 5s - 18ms/step - ia: 0.8524 - loss: 0.0943 - mae: 0.24

In [21]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
